BB-Method: Blow up and Bend
===========================

**Topic**: Path planning by deforming a single path - the BB-Method

Version | Author
------------ | -------------
0.1 | Björn Hein


This worksheet **explains the BB-Method**: instead of exploring the free space with a roadmap or a tree, one path is "bent" until it is collision-free.

License is based on Creative Commons: Attribution-NonCommercial 4.0 International (CC BY-NC 4.0) (please check: http://creativecommons.org/licenses/by-nc/4.0/)

**Prerequisite**:

* IP-4-0-Test-Environments (collision checker, benchmarks) and IP-10-0-PlanarManipulator (planar robots and their configuration space)

**Dependencies**:

* IPEnvironment.py (`CollisionChecker.penetrationDepth`), IPEnvironmentKin.py (`KinChainCollisionChecker.blowUp`), IPBBMethod.py (the planner `BBMethod`), IPVISBBMethod.py (figures of this notebook) and IPTestSuite.py (benchmarks)

Important links are:

* General Info: https://www.python.org
* NetworkX: https://networkx.org
* NumPy: https://numpy.org/doc/stable/
* Matplotlib: https://matplotlib.org/stable/
* Shapely: https://shapely.readthedocs.io/

Remember that:

* you have to press ctrl-return or shift-return, to execute the code in the code sections, only then the variables are "generated" and can be used
* you can execute the whole notebook by "Run All"

# Idea

The planners so far (PRM, RRT, A\*) explore the free space and search a path in a graph or a tree. The collision checker only answers *yes* or *no*.

The BB-Method (Boris Baginski, TU München, 1998) works differently: it starts with the straight line from $q_{start}$ to $q_{goal}$ and changes this **one path locally**, until it is collision-free. To decide which change is an improvement, it has to know **how deep** a configuration lies in an obstacle. The name describes the two ingredients:

* **B**low up the robot: rating of configurations by a (virtual) penetration depth, determined by scaling the geometry of the robot
* **B**end the trajectory: dividing and shifting path segments

The method is geometrical, heuristical and incomplete. Its goal: short search time and good path quality.

## Algorithm

Path: $P$ (list of configurations), parameters: $f_{step}$, $\delta_1$, $\delta_2$, $f_{divide}$

1. $P \leftarrow (q_{start}, q_{goal})$
2. **while** $P$ is not collision-free **do**
   1. $(q_a, q_b) \leftarrow$ segment of $P$ with the lowest rating $Q(q_{min})$
   2. divide $(q_a, q_b)$ at $q_{min}$ $\rightarrow$ $q_{new1}$, $q_{new2}$ (factor $f_{divide}$)
   3. $d \leftarrow f_{step} \cdot |q_{new2} - q_{new1}|$, limited to $[\delta_1, \delta_2]$
   4. $A \leftarrow$ shift $q_{new1}$ and/or $q_{new2}$ by $\pm d$ (cases 1 to 3)
   5. $a^* \leftarrow$ alternative in $A$ with the highest rating
   6. **if** $Q(a^*) > Q(q_{new1}, q_{new2})$ **then** replace $q_{new1}$, $q_{new2}$ by $a^*$
3. **return** $P$

If $q_{min}$ is a point of the path itself, this point is shifted instead of dividing the segment.

Only configurations **on the path** are evaluated, no C-space is computed. The following sections explain the ingredients one by one, first for a point robot in a 2D C-space (as in IP-4-0), at the end for a planar robot.

# Blow up: rating of a configuration

## Penetration depth of a point robot

The rating $Q(q)$ of a configuration has to fulfil:

* collision-free configurations get the highest rating,
* the deeper a configuration lies in an obstacle, the lower its rating.

For a point robot the C-obstacles are the obstacles themselves. A natural measure is the **penetration depth**: the distance from $q$ to the free space, i.e. how far $q$ has to be moved at least to become free. The rating is $Q(q) = -\text{depth}(q)$, free configurations get $Q = 0$.

So far the collision checker could only answer *yes* or *no*. Shapely can also compute the penetration depth, therefore `CollisionChecker` in IPEnvironment.py got a new method `penetrationDepth`:

```python
def getFreeSpace(self):
    # area inside the limits minus all obstacles (cached)
    area = box(xmin, ymin, xmax, ymax)
    return area.difference(unary_union(list(self.scene.values())))

def penetrationDepth(self, pos):
    if not self.pointInCollision(pos):
        return 0.0
    return max(self.getFreeSpace().distance(Point(pos[0], pos[1])), MIN_PENETRATION_DEPTH)
```

* `unary_union` merges all obstacles into one geometry, `difference` cuts them out of the environment.
* `distance` of a point to the free space is 0 for free points, otherwise the length of the shortest way into the free space. Configurations outside the limits are treated like configurations in an obstacle.
* A configuration on the border of an obstacle is in collision (`pointInCollision`), but its distance is 0. It gets a tiny depth (`MIN_PENETRATION_DEPTH`), so it counts as collision for the BB-Method as well.

The scene is the example of the lecture slides:

In [ ]:
import math
import matplotlib.pyplot as plt
%matplotlib inline

from IPEnvironment import CollisionChecker
import IPVISBBMethod as vis

scene, limits, start, goal = vis.lectureExample()
cc = CollisionChecker(scene, limits=limits)

testConfigurations = [[6.0, 5.0], [3.5, 3.6], [9.5, 6.0], [11.5, 3.0]]
for q in testConfigurations:
    print(q, "in collision:", cc.pointInCollision(q), "  penetration depth:", round(cc.penetrationDepth(q), 2))

The figure shows the penetration depth of every configuration (the darker, the deeper) and for the test configurations the shortest way into the free space.

**Remember**: the "landscape" is computed here only for illustration. The BB-Method itself only rates the configurations on its path - like the other planners it works in the implicit C-space.

In [ ]:
vis.showPenetrationDepth(cc, testConfigurations)

## Overlapping obstacles

Obstacles of a scene may overlap (e.g. in "MyField" of IPTestSuite.py the antennas and their heads). Then the penetration depth has to be computed for the **union** of the obstacles. Computed for every obstacle on its own (distance to its own border, maximum over all obstacles), it is much too small in the overlapping area: there the configuration is near the border of each single obstacle, but not near the free space.

Example: two overlapping boxes, left the depth per obstacle, right `penetrationDepth` of the collision checker. The marked configuration lies in both boxes.

In [ ]:
from shapely.geometry import Point, box

overlapScene = {"left": box(3, 2, 8, 6), "right": box(7, 3, 12, 8)}
ccOverlap = CollisionChecker(overlapScene, limits=[[0, 14], [0, 10]])

def depthPerObstacle(q):
    # maximum of the depths in the single obstacles (wrong for overlapping obstacles)
    p = Point(q[0], q[1])
    return max((obstacle.exterior.distance(p) for obstacle in overlapScene.values() if obstacle.contains(p)),
               default=0.0)

vis.showDepthComparison(ccOverlap, [("depth per obstacle (maximum)", depthPerObstacle),
                                    ("depth in the union (penetrationDepth)", ccOverlap.penetrationDepth)],
                        configuration=[7.5, 4.5])

In the left figure there is a "valley" along the borders inside the overlapping area. A planner following the rating would move the path into this valley - although it lies deep inside the obstacles.

## Rating of a path segment

A segment is rated by its **worst configuration** $q_{min}$, the configuration with the highest penetration depth: $Q(q_a, q_b) = Q(q_{min})$, collision-free segments get $Q = 0$. Like `lineInCollision`, the segment is checked at discrete configurations, here with a maximal distance `resolution`. The segment with the lowest rating is the **worst segment** of the path, it will be changed next.

In [ ]:
def rateConfiguration(collisionChecker, q):
    # Q(q) = -penetration depth, free: Q = 0
    return -collisionChecker.penetrationDepth(list(q))


def rateSegment(collisionChecker, qa, qb, resolution):
    # rating of a segment: (Q(q_min), q_min), q_min is its worst configuration (None if the segment is free)
    n = max(1, math.ceil(math.dist(qa, qb) / resolution))   # number of steps
    worst = (0.0, None)
    for k in range(n + 1):
        q = tuple(a + k / n * (b - a) for a, b in zip(qa, qb))
        rating = rateConfiguration(collisionChecker, q)
        if rating < worst[0]:
            worst = (rating, q)
    return worst


def ratePath(collisionChecker, path, resolution):
    # rating of a path (or a part of it): rating of its worst segment
    return min(rateSegment(collisionChecker, a, b, resolution)[0] for a, b in zip(path[:-1], path[1:]))


def pathLength(path):
    return sum(math.dist(a, b) for a, b in zip(path[:-1], path[1:]))

Two segments through the obstacle: the first one only touches it, the second one crosses it in the middle. The colored dots are the rated configurations in collision (the darker, the deeper).

In [ ]:
resolution = 0.05
segments = []
for qa, qb in [((1.0, 2.2), (6.0, 3.2)), ((4.0, 8.0), (8.0, 1.0))]:
    rating, qMin = rateSegment(cc, qa, qb, resolution)
    print(f"segment {qa} -> {qb}: Q = {rating:.2f}, q_min = {qMin}")
    segments.append((qa, qb, rating, qMin))

vis.showSegmentRatings(cc, segments, resolution)

# Bend: one step of the method

## Dividing

The worst segment $(q_a, q_b)$ is divided around its worst configuration $q_{min}$ with the partitioning factor $f_{divide} \in [0, 1]$:

$q_{new1} = q_{min} + f_{divide} \, (q_a - q_{min})$, $\quad q_{new2} = q_{min} + f_{divide} \, (q_b - q_{min})$

A small $f_{divide}$ gives a short new segment around $q_{min}$, i.e. local and small changes of the path.

In [ ]:
def divide(qa, qb, qMin, fDivide):
    # new configurations q_new1 and q_new2 around q_min
    qNew1 = tuple(m + fDivide * (a - m) for a, m in zip(qa, qMin))
    qNew2 = tuple(m + fDivide * (b - m) for b, m in zip(qb, qMin))
    return qNew1, qNew2

## Shifting

The new configurations are shifted perpendicular to the segment by the displacement $d$, in both directions:

* **case 1**: both configurations are shifted (4 combinations of the directions)
* **case 2**: only the first configuration is shifted (2 alternatives)
* **case 3**: only the second configuration is shifted (2 alternatives)

This gives $4 + 2 + 2 = 8$ alternatives. The displacement depends on the length of the new segment: $d = f_{step} \cdot |q_{new2} - q_{new1}|$, limited to $[\delta_1, \delta_2]$.

The implementation below is for 2D. In higher dimensions there are many directions perpendicular to a segment; `BBMethod` in IPBBMethod.py shifts along every vector of an orthonormal basis of them.

In [ ]:
def shiftDisplacement(qa, qb, config):
    # d = fStep * length of the segment, limited to [deltaMin, deltaMax]
    return min(config["deltaMax"], max(config["deltaMin"], config["fStep"] * math.dist(qa, qb)))


def alternatives(qa, qb, d, moveA=True, moveB=True):
    # shift qa and/or qb perpendicular to the segment by +-d (2D). Returns a list of (case, ca, cb):
    # case 1: both points, case 2: only qa, case 3: only qb
    length = math.dist(qa, qb)
    normal = (-(qb[1] - qa[1]) / length, (qb[0] - qa[0]) / length)

    def shift(q, sign):
        return (q[0] + sign * d * normal[0], q[1] + sign * d * normal[1])

    result = []
    if moveA and moveB:
        result += [(1, shift(qa, signA), shift(qb, signB)) for signA in (1, -1) for signB in (1, -1)]
    if moveA:
        result += [(2, shift(qa, sign), qb) for sign in (1, -1)]
    if moveB:
        result += [(3, qa, shift(qb, sign)) for sign in (1, -1)]
    return result

## Evaluation and replacement

Every alternative replaces the two configurations in the changed part of the path ($q_{prev}$, $q_{new1}$, $q_{new2}$, $q_{next}$) and is rated by its worst segment. The resulting segments do not have to be collision-free, they are improved in the following steps.

* The **best** alternative has the highest rating, among equal ratings it is the shortest one.
* It **replaces** the new configurations only if its rating is higher than the rating of the divided path. Otherwise only the division is kept: in the next step the worst segment is shorter, so $d$ gets smaller (down to $\delta_1$) - the method searches more locally.

The following function performs one step. Besides the new path it returns all intermediate results, they are used for the figures.

In [ ]:
def bbStep(collisionChecker, path, config, eps=1e-9):
    # one step of the BB-Method: None if the path is collision-free, otherwise a dict (new path: step["path"])
    resolution = config["resolution"]

    # 1. worst segment (q_a, q_b) and its worst configuration q_min
    ratings = [rateSegment(collisionChecker, a, b, resolution) for a, b in zip(path[:-1], path[1:])]
    i = min(range(len(ratings)), key=lambda k: ratings[k][0])
    rating, qMin = ratings[i]
    if rating >= 0.0:
        return None
    qa, qb = path[i], path[i + 1]

    # 2. divide at q_min - if q_min is a point of the path (not start or goal), this point is shifted instead
    atA = math.dist(qMin, qa) < eps and i > 0
    atB = math.dist(qMin, qb) < eps and i + 1 < len(path) - 1
    if atA or atB:
        dividedPath, ia, ib, divided = list(path), i, i + 1, False
        moveA, moveB = atA, not atA
    else:
        qNew1, qNew2 = divide(qa, qb, qMin, config["fDivide"])
        dividedPath, ia, ib, divided = path[:i + 1] + [qNew1, qNew2] + path[i + 1:], i + 1, i + 2, True
        moveA, moveB = True, True

    # 3. shift: alternatives for the changed part q_prev, q_new1, q_new2, q_next of the path
    d = shiftDisplacement(dividedPath[ia], dividedPath[ib], config)
    lo, hi = max(ia - 1, 0), min(ib + 1, len(dividedPath) - 1)
    current = dividedPath[lo:hi + 1]
    currentRating = ratePath(collisionChecker, current, resolution)
    candidates = []
    for case, ca, cb in alternatives(dividedPath[ia], dividedPath[ib], d, moveA, moveB):
        subPath = list(current)
        subPath[ia - lo], subPath[ib - lo] = ca, cb
        candidates.append(dict(case=case, ca=ca, cb=cb, subPath=subPath, length=pathLength(subPath),
                               rating=ratePath(collisionChecker, subPath, resolution)))

    # 4. evaluate: highest rating, among equal ratings the shortest
    best = max(candidates, key=lambda a: (round(a["rating"], 9), -a["length"]))

    # 5. replace, if the best alternative is better than the divided path
    newPath = list(dividedPath)
    replaced = best["rating"] > currentRating
    if replaced:
        newPath[ia], newPath[ib] = best["ca"], best["cb"]
    return dict(previous=list(path), segment=i, qa=qa, qb=qb, qMin=qMin, rating=rating, divided=divided,
                dividedPath=dividedPath, ia=ia, ib=ib, d=d, currentRating=currentRating,
                alternatives=candidates, best=best, replaced=replaced, path=newPath)

The first step on the example of the lecture with the parameters of the slides:

In [ ]:
config = dict()
config["fStep"] = 0.8         # d = fStep * length of the new segment ...
config["deltaMin"] = 0.2      # ... limited to [delta_1, delta_2]
config["deltaMax"] = 2.0
config["fDivide"] = 0.3       # partitioning factor
config["resolution"] = 0.02   # maximal distance of the rated configurations on a segment
config["maxIterations"] = 40

step = bbStep(cc, [tuple(start), tuple(goal)], config)
print(f"worst segment: Q(q_min) = {step['rating']:.2f}, d = {step['d']:.2f}")
for alternative in step["alternatives"]:
    print(f"  case {alternative['case']}: Q = {alternative['rating']:6.2f}, length = {alternative['length']:.2f}")
print("replaced:", step["replaced"])

vis.showStep(cc, step, config)

None of the eight alternatives is collision-free (dashed). The best one (blue) still touches the obstacle, but its rating is higher than the rating of the divided path, so it replaces the new configurations. The following steps continue with the new worst segment.

# The complete algorithm

The steps are repeated until the whole path is collision-free (or a maximal number of iterations is reached):

In [ ]:
def planBB(collisionChecker, start, goal, config):
    # BB-Method: bend the straight line from start to goal until it is collision-free
    # returns (path, steps, free)
    path = [tuple(start), tuple(goal)]
    steps = []
    for iteration in range(config["maxIterations"]):
        step = bbStep(collisionChecker, path, config)
        if step is None:
            return path, steps, True   # all segments are collision-free
        steps.append(step)
        path = step["path"]
    return path, steps, ratePath(collisionChecker, path, config["resolution"]) >= 0.0

In [ ]:
path, steps, free = planBB(cc, start, goal, config)
print(f"collision-free: {free}, {len(steps)} steps, {len(path)} configurations, path length {pathLength(path):.2f}")
print("straight line:", round(math.dist(start, goal), 2))

vis.showSteps(cc, steps, path)

**Try it** (optional, needs `ipywidgets`): step through the whole run. Every step consists of five phases: worst segment, dividing, shifting, evaluation, result.

In [ ]:
from ipywidgets import interact, IntSlider

frames = vis.stepFrames(steps)
interact(lambda frame: vis.plotFrame(cc, steps, path, frames[frame], config),
         frame=IntSlider(min=0, max=len(frames) - 1, value=0));

## The planner BBMethod

IPBBMethod.py contains the same algorithm as class `BBMethod` with the interface of the other planners: `planPath(startList, goalList, config)` returns the path as list of nodes of `planner.graph` (empty if no collision-free path was found). The steps are recorded in `planner.steps`, the configurations of the path are in `planner.path`. Additionally the ratings of the segments are cached and the shifting works in any dimension.

With the same parameters it finds exactly the same path:

In [ ]:
from IPBBMethod import BBMethod
from IPVISBBMethod import bbVisualize

planner = BBMethod(cc)
solution = planner.planPath([start], [goal], config)
print("solution:", solution)
print("same path as planBB:", [tuple(q) for q in planner.path] == path)

fig, ax = plt.subplots(figsize=(7, 5))
bbVisualize(planner, solution, ax=ax)
vis.setupAxes(ax, cc, "BBMethod from IPBBMethod.py, visualized with bbVisualize")

# Heuristic parameters

* **$f_{step}$, $\delta_1$, $\delta_2$**: the displacement $d$. A small $d$ needs many steps to get out of an obstacle, a large $d$ makes large detours (and may jump over thin obstacles, as only the rated configurations are checked).
* **$f_{divide}$**: small values change the path only locally around $q_{min}$, large values change long parts of the path.

The number of rated configurations (calls of `penetrationDepth`, counted by `IPPerfMonitor`, see IP-4-0) is a measure for the planning effort.

In [ ]:
import pandas as pd
from IPPerfMonitor import IPPerfMonitor

def runBB(collisionChecker, startList, goalList, config):
    # one run of BBMethod; returns (planner, solution, statistics)
    IPPerfMonitor.clearData()
    planner = BBMethod(collisionChecker)
    solution = planner.planPath(startList, goalList, config)
    calls = IPPerfMonitor.dataFrame()
    statistics = {"free": bool(solution), "steps": len(planner.steps),
                  "replaced": sum(step["replaced"] for step in planner.steps),
                  "configurations on path": len(planner.path), "path length": round(pathLength(planner.path), 2),
                  "rated configurations": int((calls["name"] == "penetrationDepth").sum()),
                  "time [s]": round(float(calls[calls["name"] == "planPath"]["time"].sum()), 3)}
    return planner, solution, statistics

variants = {"slides": {},
            "fDivide = 0.6": {"fDivide": 0.6},
            "fDivide = 0.1": {"fDivide": 0.1},
            "fStep = 0.5": {"fStep": 0.5},
            "deltaMax = 1.0": {"deltaMax": 1.0}}
rows, results = [], []
for name, change in variants.items():
    variantConfig = {**config, "maxIterations": 200, **change}
    variantPlanner, variantSolution, statistics = runBB(cc, [start], [goal], variantConfig)
    rows.append({"variant": name, **statistics})
    results.append((name, variantPlanner, variantSolution))

vis.showResults(results, columns=len(results), panelWidth=3.6)
pd.DataFrame(rows).set_index("variant")

Observations (maximal 200 steps):

* A larger $f_{divide}$ changes longer parts of the path: fewer steps, but the path gets longer.
* With a very small $f_{divide}$ the changes are so local that the method gets stuck.
* A smaller $f_{step}$ needs more steps, here the path even gets a bit shorter.
* A small $\delta_2$ needs many small steps; the path is bent again and again and gets very long.

# Benchmarks

`BBMethod` on the benchmarks of IPTestSuite.py, once with the parameters of the slides and once with a larger displacement ($\delta_1 = 0.5$, $\delta_2 = 5$). Unlike the PRM and the RRT, the BB-Method is deterministic: every run gives the same result.

In [ ]:
import IPTestSuite as ts

benchConfigs = {"slides": {**config, "resolution": 0.05, "maxIterations": 100},
                "large d": {**config, "resolution": 0.05, "maxIterations": 100, "deltaMin": 0.5, "deltaMax": 5.0}}
rows = []
for configName, benchConfig in benchConfigs.items():
    results = []
    for benchmark in ts.benchList:
        benchPlanner, benchSolution, statistics = runBB(benchmark.collisionChecker, benchmark.startList,
                                                        benchmark.goalList, benchConfig)
        rows.append({"benchmark": benchmark.name, "parameters": configName, **statistics})
        results.append((f"{configName}: {benchmark.name}\n" + ("free" if benchSolution else "no free path")
                        + f" after {len(benchPlanner.steps)} steps", benchPlanner, benchSolution))
    vis.showResults(results, panelWidth=4.2)

pd.DataFrame(rows).set_index(["benchmark", "parameters"])

Observations:

* "MyField" is solved in two steps, the path stays close to the straight line.
* With the parameters of the slides "Bottleneck" is not solved: inside the long wall the penetration depth is the same everywhere (half of its thickness). Shifting along the wall does not improve the rating - a **plateau**. Only a displacement, which reaches the passage, helps: with $\delta_2 = 5$ the path gets around the end of the wall in three steps.
* In "Fat bottleneck" the path reaches the passage with the parameters of the slides, but gets stuck at one of its corners. Again the larger displacement solves it.
* "Trap" is not solved with any of the parameters: the path ends in a **local minimum**, see below.
* After getting stuck the method only divides the path ("replaced" counts the steps with replacement), until the maximal number of iterations is reached.

## Local minimum

In "Trap" the start lies inside the "U", the straight line to the goal runs through its bottom. The path cannot leave the "U" by small local changes: every shift of the configurations in the bottom keeps the path inside the wall or makes it deeper. No alternative is better, the method only divides further and the configurations pile up in the wall:

In [ ]:
trap = ts.benchList[0]
trapPlanner, trapSolution, statistics = runBB(trap.collisionChecker, trap.startList, trap.goalList,
                                              benchConfigs["slides"])
print(statistics)
vis.showLocalMinimum(trapPlanner, benchConfigs["slides"])

The BB-Method only uses local information along the path. It has no memory of the explored space and cannot "step back". Such local minima are the price for the speed; in practice the method is combined with a global planner, which provides intermediate configurations.

# Robots: blow up

For a point robot the penetration depth is a distance in the C-space. For a robot with an extent this distance is unknown: the C-obstacles are not computed. Baginski rates a configuration with the robot geometry in the work space instead (**blow up**):

* every link is shrunk to its **skeleton** and blown up again link by link from the base,
* $i_{coll}$: the first link, which collides,
* $scale_{i_{coll}} \in [0, 1]$: the largest scale of this link without collision,
* $Q_{scale}(q) = (i_{coll} - 1) + scale_{i_{coll}} \in [0, \dim(C)]$, collision-free: $Q_{scale}(q) = \dim(C)$.

The earlier the colliding link in the kinematic chain, the lower the rating. For the BB-Method the rating is converted into a penetration depth: $\dim(C) - Q_{scale}(q)$, collision-free configurations get 0.

`KinChainCollisionChecker` (IPEnvironmentKin.py, see IP-10-0) got the needed extensions:

* `link_radius`: the links are capsules with this radius around the line between the joints (default 0: only lines as before),
* `blowUp(pos)`: returns $(i_{coll}, scale_{i_{coll}})$, `qScale(pos)` the rating and `penetrationDepth(pos)` the depth for the BB-Method,
* `skeleton="axis"`: the skeleton of a link is the line between its joints (its axis), only the radius is blown up. Then $scale_{i_{coll}}$ is the distance of the axis to the obstacles divided by `link_radius` - Shapely computes it directly.

Example: a planar robot with two links (length 1.5, radius 0.15) and a box. Red: the colliding link blown up as far as possible, dashed: its real size.

In [ ]:
from shapely.geometry import box
from IPPlanarManipulator import PlanarRobot
from IPEnvironmentKin import KinChainCollisionChecker

robot = PlanarRobot(n_joints=2)
robotScene = {"box": box(1.15, 1.7, 1.85, 2.3)}
jointLimits = [[-math.pi, math.pi], [-math.pi, math.pi]]
robotStart, robotGoal = [-0.45, -0.95], [0.6, 1.5]

ccAxis = KinChainCollisionChecker(robot, robotScene, limits=jointLimits, link_radius=0.15, skeleton="axis")

robotConfigurations = [[0.4, 0.6], [0.5, 0.5], [0.3, 1.0]]
for q in robotConfigurations:
    print(q, "blowUp:", ccAxis.blowUp(q), " Q_scale:", round(ccAxis.qScale(q), 2),
          " penetration depth:", round(ccAxis.penetrationDepth(q), 2))

vis.showBlowUp(ccAxis, robotConfigurations)

The rating of all configurations in the joint space (again only for illustration, the colors start at $Q_{scale} = 1$). The obstacle can only be reached by the second link, so $Q_{scale} \ge 1$. The BB-Method starts with the straight line between the two joint configurations:

In [ ]:
robotConfig = dict(fStep=0.8, deltaMin=0.1, deltaMax=1.0, fDivide=0.3, resolution=0.03, maxIterations=60)

axisPlanner, axisSolution, statistics = runBB(ccAxis, [robotStart], [robotGoal], robotConfig)
print(statistics)
vis.showRobotPath(ccAxis, axisPlanner.path, title='skeleton = "axis": no collision-free path')

The method gets stuck immediately (only dividing, no replacement). The reason is visible in the landscape: as soon as the **axis** of the second link collides, $scale = 0$ and $Q_{scale} = 1$ - the same value in the whole inner part of the C-obstacle (**plateau**). There every alternative has the same worst rating as the current path, so no alternative is better. A rating for the BB-Method has to tell, which direction is better - also deep inside an obstacle.

## Variant: shrink a link to its joint

With `skeleton="joint"` the skeleton of a link is the joint at its base, the **whole link** (length and radius) is scaled from there. The joint is always free, if the previous link is free, so $scale_{i_{coll}} > 0$: it is the part of the link, which is free. It changes continuously with the configuration, the plateau disappears. (The largest scale is found by bisection.)

In [ ]:
ccJoint = KinChainCollisionChecker(robot, robotScene, limits=jointLimits, link_radius=0.15, skeleton="joint")

for q in robotConfigurations:
    print(q, "blowUp:", ccJoint.blowUp(q), " Q_scale:", round(ccJoint.qScale(q), 2))

vis.showBlowUp(ccJoint, robotConfigurations)

In [ ]:
jointPlanner, jointSolution, statistics = runBB(ccJoint, [robotStart], [robotGoal], robotConfig)
print(statistics)
vis.showRobotPath(ccJoint, jointPlanner.path, title='skeleton = "joint": collision-free path')

Now the rating increases towards the upper end of the C-obstacle: there the box touches only the tip of the second link. The method bends the path around this end. In the work space: the robot first folds its second link and then turns it past the box.

**Animation** (optional, takes some seconds): the function `animateSolution` of IPEnvironmentKin.py works with every planner, which provides `graph` and a solution (the robot is drawn as lines).

In [ ]:
from IPEnvironmentKin import animateSolution

animateSolution(jointPlanner, ccJoint, jointSolution, bbVisualize)

## Example: planar robot with 10 links

The BB-Method was developed for robots with **many degrees of freedom**. A planar robot with 10 links (length 0.4, radius 0.08) has a 10-dimensional C-space. It cannot be drawn any more, and a grid as in the figures above would need $120^{10}$ configurations. The BB-Method only rates the configurations along its path. In 10 dimensions there are 9 directions perpendicular to a segment, so `BBMethod` creates $9 \cdot 8 = 72$ alternatives per step.

The task: the robot points upwards and has to move to a curled pose on the lower left. A block stands in its way: the straight line in the joint space turns the robot right through the block (orange: in collision).

In [ ]:
robot10 = PlanarRobot(n_joints=10)
for joint in robot10.joints:
    joint.a = 0.4   # length of every link

scene10 = {"block": box(1.4, -1.5, 2.2, 1.0)}
cc10 = KinChainCollisionChecker(robot10, scene10, limits=[[-math.pi, math.pi]] * 10, link_radius=0.08,
                                skeleton="joint", fk_resolution=0.05)
start10 = [1.8] + [-0.15] * 9   # first joint, then the 9 other joints
goal10 = [-0.7] + [-0.3] * 9

vis.showRobotMotions(cc10, [("straight line in the joint space", [start10, goal10])])

The BB-Method with the parameters of the slides and with a larger partitioning factor $f_{divide} = 0.5$:

In [ ]:
configs10 = {"slides": {**config, "resolution": 0.05, "maxIterations": 60},
             "fDivide = 0.5": {**config, "resolution": 0.05, "maxIterations": 60, "fDivide": 0.5}}
rows, motions = [], [("straight line", [start10, goal10])]
for name, config10 in configs10.items():
    planner10, solution10, statistics = runBB(cc10, [start10], [goal10], config10)
    rows.append({"parameters": name, **statistics})
    motions.append((f"{name}: " + ("collision-free" if solution10 else "no free path")
                    + f" after {len(planner10.steps)} steps", planner10.path))
print("alternatives per step:", len(planner10.steps[0]["alternatives"]))
print("length of the straight line:", round(math.dist(start10, goal10), 2))

vis.showRobotMotions(cc10, motions)
pd.DataFrame(rows).set_index("parameters")

In both runs the robot rolls itself up, passes the block on the inside and unrolls again. With $f_{divide} = 0.5$ longer parts of the path are changed at once: 4 instead of 18 steps and a shorter path. Even in 10 dimensions only some thousand configurations are rated.

The last run step by step:

In [ ]:
vis.showRobotFilmStrip(cc10, planner10.path)

**Animation** (optional, takes some seconds):

In [ ]:
animateSolution(planner10, cc10, solution10, bbVisualize, workSpaceLimits=[[-4.2, 4.2], [-4.2, 4.2]])

## Comparison with the Lazy-PRM

The same task with the Lazy-PRM (see IP-7-0, IPLazyPRM.py): it builds a random roadmap without any collision checks and checks only the nodes and edges of the shortest path in the roadmap, until this path is collision-free. Its edges are checked with the same resolution as the BB-Method rates its segments (`fk_resolution=0.05` above). The Lazy-PRM is randomized, therefore it is run 20 times (with fixed seeds, so the table is reproducible).

Effort: for the BB-Method the rated configurations (calls of `penetrationDepth`), for the Lazy-PRM the checked configurations (calls of `pointInCollision`), both counted by `IPPerfMonitor`.

In [ ]:
import random
import numpy as np
from IPLazyPRM import LazyPRM

lazyConfig10 = dict()
lazyConfig10["initialRoadmapSize"] = 50   # number of nodes of the first roadmap
lazyConfig10["updateRoadmapSize"] = 20    # number of nodes to add, if there is no connection from start to goal
lazyConfig10["kNearest"] = 8              # number of nodes to connect to
lazyConfig10["maxIterations"] = 40        # number of updates of the roadmap

def runLazy(collisionChecker, startList, goalList, config, seed):
    # one run of the Lazy-PRM; returns (planner, solution, path as list of configurations, statistics)
    random.seed(seed)
    IPPerfMonitor.clearData()
    planner = LazyPRM(collisionChecker)
    solution = planner.planPath(startList, goalList, config)
    calls = IPPerfMonitor.dataFrame()
    path = [planner.graph.nodes[node]["pos"] for node in solution]
    statistics = {"free": bool(solution), "configurations on path": len(path),
                  "path length": round(pathLength(path), 2) if path else np.nan,
                  "checked configurations": int((calls["name"] == "pointInCollision").sum()),
                  "time [s]": round(float(calls[calls["name"] == "planPath"]["time"].sum()), 3)}
    return planner, solution, path, statistics

lazyRuns = [runLazy(cc10, [start10], [goal10], lazyConfig10, seed) for seed in range(20)]
lazyStatistics = pd.DataFrame([run[3] for run in lazyRuns])
bbPlanner, bbSolution, bbStatistics = runBB(cc10, [start10], [goal10], configs10["fDivide = 0.5"])

random.seed(0)
samples = [[random.uniform(-math.pi, math.pi) for _ in range(10)] for _ in range(1000)]
print(f"free random configurations: {sum(not cc10.pointInCollision(q) for q in samples) / len(samples):.0%}")
print("length of the straight line:", round(math.dist(start10, goal10), 2))

pd.DataFrame([
    {"planner": "BB-Method (fDivide = 0.5)", "runs": 1, "success rate": float(bbStatistics["free"]),
     "path length": bbStatistics["path length"], "checked configurations": bbStatistics["rated configurations"],
     "time [s]": bbStatistics["time [s]"]},
    {"planner": "Lazy-PRM (mean)", "runs": len(lazyRuns), "success rate": lazyStatistics["free"].mean(),
     "path length": lazyStatistics["path length"].mean(),
     "checked configurations": lazyStatistics["checked configurations"].mean(),
     "time [s]": lazyStatistics["time [s]"].mean()}]).set_index("planner").round(2)

The motions of the BB-Method and of a typical run of the Lazy-PRM (the run with the median path length):

In [ ]:
lengths = lazyStatistics["path length"].fillna(np.inf)
typical = int(lengths.sort_values().index[len(lengths) // 2])
lazyPath = lazyRuns[typical][2]

vis.showRobotMotions(cc10, [("BB-Method", bbPlanner.path),
                            (f"Lazy-PRM (run {typical}): {len(lazyPath)} configurations on the path", lazyPath)],
                     poses=16)
vis.showRobotFilmStrip(cc10, lazyPath)

Observations:

* The Lazy-PRM finds a path in every run - with fewer checked configurations and in less time than the BB-Method. The reason: most of the C-space of this robot is free, so most of the random edges are free as well. Moreover a collision check is cheaper than a rating (blow up with bisection).
* But the path of the Lazy-PRM leads via random configurations: in the joint space it is several times longer than the path of the BB-Method, the robot swings around wildly. (Self-collisions of the links are not checked by `KinChainCollisionChecker`, therefore the robot may even cross itself.) The path of the BB-Method stays close to the straight line - **path quality** is the strength of the BB-Method.
* Paths of sampling-based planners are therefore smoothed afterwards (next chapter). The BB-Method is deterministic: it gives the same path in every run.

# Properties of the BB-Method

* **Fast**: only local evaluations along the path, no explicit C-space (see the number of rated configurations). In a mostly free C-space a sampling-based planner can be even faster - with much longer paths (see the comparison with the Lazy-PRM).
* **Many DOF**: $Q_{scale}$ works directly on the robot geometry, its effort grows only linearly with the number of links. The shifting works in any dimension (robot with 10 links: 72 alternatives per step).
* **Good path quality**: the path stays close to the straight line (compare the paths of the Lazy-PRM).
* **Heuristical and incomplete**: may get stuck in a local minimum (see "Trap"). The success depends on the rating as well: it must not have plateaus (see the robot example).
* The parameters $f_{step}$, $\delta_1$, $\delta_2$, $f_{divide}$ influence success and path quality.
* The path runs close to the obstacles: smoothing or a safety distance afterwards.

# Experiments and Exercises

1. Find parameters, with which the bottlenecks are solved with as few rated configurations as possible. Why does a larger $\delta_2$ help here?
2. Change the tie-break in `bbStep`: take the first alternative with the highest rating instead of the shortest one. How does the path change?
3. **Safety distance**: rate configurations with a penetration depth into the obstacles grown by a distance $s$ (`buffer`). Which path results for the example of the lecture?
4. Use `BBMethod` with a planar robot with three joints (`PlanarRobot(n_joints=3)`, three joint limits). How many alternatives are created in every step?
5. Implement `penetrationDepth` for `CollisionCheckerShapeRobot` (IPEnvironmentShapeRobot.py, see IP-10-2): (a) blow up the robot around its reference point (`shapely.affinity.scale`) or (b) use the overlap area of robot and obstacles (`intersection(...).area`). Which measure has plateaus?
6. Robot with 10 links: find start and goal configurations, for which the BB-Method fails. Why does it fail? (Hint: can the robot get to the other side of the block by small local changes?)
7. Compare the BB-Method and the Lazy-PRM on the robot with 10 links after adding further obstacles: how do the success rates and the numbers of checked configurations change?
8. (Advanced) Escape from local minima: if the rating did not improve for some steps, insert an intermediate configuration (e.g. found by an RRT) into the path.